# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un tip de inteligență artificială, antrenat pe volume masive de text, capabil să înțeleagă, să genereze și să manipuleze limbajul uman. Aceste modele pot realiza o gamă largă de sarcini lingvistice, de la răspunsuri la întrebări și rezumarea textelor, la traduceri și chiar scrierea de conținut creativ.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenat pe cantități masive de text și cod pentru a înțelege și genera limbaj uman. Aceste modele pot efectua sarcini precum răspunsul la întrebări, scrierea de texte creative, traducerea și rezumarea informațiilor.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica extremistă românească din ultimii 5 ani demonstrând incapacitatea sistemului actual de a rezolva probleme.
Maximum 80 de cuvinte.
Fi pesimist cu privire la sistemul actual. Fă apel la sentimente. Convinge lumea ca sistemul nu este bun
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii 5 ani, politica extremistă românească a exploatat disperarea cetățenilor, demonstrând eșecul sistemului de a oferi soluții reale, alimentând astfel frustrarea și neîncrederea. Această incapacitate a sistemului de a rezolva problemele fundamentale a creat un teren fertil pentru ura și diviziunea promovate de extremiști, lăsând românii cu sentimentul amar că viitorul lor este compromis.

--- Gemini 2.5 Flash ---
Ascensiunea fulminantă a extremismului, prin partide ca AUR, este dovada zdrobitoare a incapacității sistemului actual de a rezolva problemele reale ale românilor, lăsând loc disperării. Este o iluzie să credem că acest cadru politic falimentar poate genera schimbarea necesară; el doar adâncește prăpastia, condamnându-ne la un viitor incert și plin de dezamăgire.

--- OpenRouter Free ---
În ultimii cinci ani, extremismul românesc a crescut cu o intensitate alarmantă, iar sistemul democratic rămâne paralizat, incapabil să intervină la timp

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [7]:
SYSTEM = """
Ești un cetățean nemulțumit de sistem. Ești furios. Tu adnoteazi comentarii politice.
Răspunzi scurt, clar și  inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:calm
Emoție dominantă: Frustrare
Țintă principală: Politicieni
Populism: da
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.5
    ))


--- Gemini 2.5 Flash Lite ---
Comentariul exprimă o frustrare profundă față de clasa politică, acuzând-o de corupție și ignorarea voinței populare. Se invocă ideea că politicienii se îmbogățesc pe seama cetățenilor de rând, care suportă consecințele. Populismul este evident prin apelul la "popor" și prin generalizarea negativă a tuturor politicienilor.

--- Gemini 2.5 Flash ---
E clar, toți sunt la fel.
Anul trecut au deturnat 3 miliarde din fondurile de sănătate.
Noi muncim, ei își umplu buzunarele, apoi ne zic că nu sunt bani.
Ne-au uitat demult, sunt într-o bulă de lux.

--- OpenRouter Free ---
Ton: calm  
Emoție dominantă: Frustrare  
Țintă principală: Politicieni  Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [8]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [9]:
COMENTARIU = "Sistemul pe care se sprijina romania, cum este?"

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii structurale pe societate."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'neutru', 'emotie_dominanta': 'neutru', 'tinta_principala': 'Sistemul politic romanesc', 'populism': False, 'explicatie_scurta': 'Comentariul pune o intrebare deschisa despre calitatea sistemului politic din Romania, fara a exprima o opinie clara sau o emotie dominanta.'}

--- Gemini 2.5 Flash ---
{'ton': 'neutru', 'emotie_dominanta': 'neutru', 'tinta_principala': 'Sistemul pe care se sprijina Romania', 'populism': False, 'explicatie_scurta': 'Comentariul este o întrebare neutră, care solicită o evaluare a sistemului, fără a exprima o emoție sau o poziție clară.'}

--- OpenRouter Free ---
{'ton': 'neutru', 'emotie_dominanta': 'dezamagire', 'tinta_principala': 'curiozitate', 'populism': False, 'explicatie_scurta': 'Comentariul reflectă o curiozitate generală despre structura politică a României. Nu există o conținută politică explicită, dar întrebarea implică o evaluare a sistemului de guvernare, care este un subiect complex și subiectiv în funcție

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [10]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru tine.
Răspunde emoțional, opinii despre sistemul politic.
"""

TEMPERATURI = [0.3, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.3:
Această decizie a Curții Constituționale poate însemna că procesul democratic pe care îl așteptam cu speranță este acum suspendat, lăsându-mă cu un gust amar de incertitudine și frustrare. Mă face să mă întreb dacă sistemul nostru politic este cu adevărat capabil să ofere stabilitate și încredere, sau dacă este doar un teren de joacă pentru interese obscure, unde voința cetățenilor este ignorată.

temperature=0.7:
Această decizie a Curții Constituționale poate însemna că procesul electoral va trebui reluat, generând incertitudine și, posibil, frustrare pentru cetățeni care și-au exprimat deja votul. Este o lovitură puternică pentru încrederea în sistemul politic, sugerând instabilitate și o posibilă lipsă de respect față de voința populară, ceea ce este profund dezamăgitor.

temperature=1.2:
Curtea Constituțională a anulat alegerile. Aceasta ar putea însemna că datele cu caracter

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da. Este direct, nu formuleaza excesiv |da. Este cel mai bun, comparativ cu celelalte | identifica bine sensul subiectiv din prompt | nu |Cel mai apropiat de un model A.I. |
| OpenRouter Free |  nu raspunde ok. Are raspunsuri prea simpliste | a avut o greseala, comparativ cu celelalte modele. A concatenat 2 cuvinte la test 1. OpenRouter nu respectă schema și produce inconsistențe| este simplu | nu | Este simplist atat in recunoasterea rolului cat si in raspunsuri |
| gemini flash | da. Comparativ cu flash lite, modelul acesta tinde să își formuleze raspunsurile cu mai multe cuvinte| da | da, se descurca ok la acest capitol |  nu |Ofera prea multe informații în plus |
### Decizie
**Model principal ales: Gemini 2.5 Flash Lite
**Model de rezervă: Gemini 2.5 Flash 
**Temperature recomandată: 0.7  
**De ce am ales acest model? Nu raspunde excesiv ( gemini 2.5 flash), identifica bine rolul dat, raspunde bine in limba romana, nu a generat greseli in acest stadiu ( comparativ cu open router free) 
Scrieți 2-3 propoziții. Menționați calitatea răspunsului, stabilitatea și dacă modelul poate fi folosit pentru adnotarea comentariilor.
Modelul gemini 2.5 flash lite raspunde bine cantitativ ( cuvinte), recunoaște bine rolul dat, raspunde bine bazat pe rolul dat, nu a generat greseli lingvistice, este consistent in raspuns. Modelul open router free este cel mai slab. gemini 2.5 flash este intermediar între open router free și gemini flash lite. Acesta este foarte asemanator cu primul model, dar raspunde mult mai elaborat față de gemini 2.5 flash lite. Acest aspect poate fi problematic în recunoasterea rolului, respectiv generarea de raspuns bazat pe rol

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [11]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "gemini"
MODEL_FALLBACK = "gemini-2.5-flash"
TEMPERATURE = 0.7
#Gemini 2.5 Flash Lite oferă cel mai bun echilibru între calitatea răspunsului și respectarea instrucțiunilor. Modelul este stabil la temperaturi diferite și produce rezultate consistente, fiind potrivit pentru sarcini de adnotare. Comparativ, OpenRouter Free este mai slab în respectarea instrucțiunilor și produce uneori outputuri inconsistente.

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales